In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

PREFIX = "wdatt_movie_"


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

PREFIX = "wdatt_movie_"

bronze_movies  = spark.table(f"{PREFIX}bronze_movies")
bronze_links   = spark.table(f"{PREFIX}bronze_links")
bronze_tags    = spark.table(f"{PREFIX}bronze_tags")
bronze_ratings = spark.table(f"{PREFIX}bronze_ratings_top500")
bronze_scraped = spark.table(f"{PREFIX}bronze_scraped_metadata")  # ✅ defines bronze_scraped

print("✅ Bronze loaded:",
      bronze_movies.count(),
      bronze_links.count(),
      bronze_tags.count(),
      bronze_ratings.count(),
      bronze_scraped.count())


In [0]:
def safe_int(colname: str):
    return F.expr(f"try_cast(nullif(trim({colname}), '') as int)")

def safe_long(colname: str):
    return F.expr(f"try_cast(nullif(trim({colname}), '') as bigint)")

def safe_double(colname: str):
    return F.expr(f"try_cast(nullif(trim({colname}), '') as double)")


In [0]:
silver_movies = (
    bronze_movies
    .select(
        safe_int("movieId").alias("movieId"),
        F.col("title").cast("string").alias("title"),
        F.col("genres").cast("string").alias("genres")
    )
    .where(F.col("movieId").isNotNull())
    .withColumn("movie_year_raw", F.regexp_extract("title", r"\((\d{4})\)$", 1))
    .withColumn("movie_year", F.expr("try_cast(nullif(movie_year_raw,'') as int)"))
    .drop("movie_year_raw")
    .withColumn("title_clean", F.regexp_replace("title", r"\s*\(\d{4}\)$", ""))
    .withColumn(
        "genres_array",
        F.when((F.col("genres").isNull()) | (F.col("genres") == "(no genres listed)"), F.array().cast("array<string>"))
         .otherwise(F.split("genres", r"\|"))
    )
    .dropDuplicates(["movieId"])
)

silver_movies.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}silver_movies")
print("silver_movies:", silver_movies.count())


In [0]:
silver_links = (
    bronze_links
    .select(
        safe_int("movieId").alias("movieId"),
        safe_int("imdbId").alias("imdbId"),
        safe_int("tmdbId").alias("tmdbId"),
    )
    .where(F.col("movieId").isNotNull())
    .dropDuplicates(["movieId"])
)

silver_links.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}silver_links")
print("silver_links:", silver_links.count())


In [0]:
silver_ratings = (
    bronze_ratings
    .select(
        safe_int("userId").alias("userId"),
        safe_int("movieId").alias("movieId"),
        safe_double("rating").alias("rating"),
        safe_long("timestamp").alias("timestamp"),
    )
    .where(F.col("userId").isNotNull() & F.col("movieId").isNotNull() & F.col("rating").isNotNull())
    .withColumn("rating_ts", F.to_timestamp(F.from_unixtime("timestamp")))
)

silver_ratings.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}silver_ratings")
print("silver_ratings:", silver_ratings.count())


In [0]:
silver_tags = (
    bronze_tags
    .select(
        safe_int("userId").alias("userId"),
        safe_int("movieId").alias("movieId"),
        F.col("tag").cast("string").alias("tag"),
        safe_long("timestamp").alias("timestamp"),
    )
    .where(F.col("userId").isNotNull() & F.col("movieId").isNotNull() & F.col("tag").isNotNull())
    .withColumn("tag_ts", F.to_timestamp(F.from_unixtime("timestamp")))
)

silver_tags.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}silver_tags")
print("silver_tags:", silver_tags.count())


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

PREFIX = "wdatt_movie_"
bronze_scraped = spark.table(f"{PREFIX}bronze_scraped_metadata")

def safe_int(colname: str):
    return F.expr(f"try_cast(nullif(trim({colname}), '') as int)")

cols = set(bronze_scraped.columns)

if "director" in cols:
    director_expr = F.col("director").cast("string")
elif "directors" in cols:
    director_expr = F.element_at(F.col("directors"), 1).cast("string")
else:
    director_expr = F.lit(None).cast("string")

budget_clean = F.regexp_replace(F.col("budget").cast("string"), r"[\$,]", "").cast("double")

silver_scraped = (
    bronze_scraped
    .withColumn("movieId", safe_int("movieId"))
    .withColumn("director", director_expr)
    .withColumn("budget_clean", budget_clean)
    .withColumn("scraped_title", F.col("title").cast("string"))
    .withColumn("status", F.col("status").cast("string") if "status" in cols else F.lit("ok"))
    .withColumn("poster_url", F.col("poster_url").cast("string") if "poster_url" in cols else F.lit(None).cast("string"))
    .dropna(subset=["movieId"])
)

silver_scraped = (
    silver_scraped
    .withColumn("status_rank", F.when(F.col("status") == "ok", F.lit(1)).otherwise(F.lit(2)))
    .withColumn("rn", F.row_number().over(Window.partitionBy("movieId").orderBy(F.col("status_rank").asc_nulls_last())))
    .where(F.col("rn") == 1)
    .drop("rn","status_rank")
)

silver_scraped.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}silver_scraped_metadata")
print("✅ silver_scraped rows:", silver_scraped.count())


In [0]:
silver_movie_master = (
    silver_movies
    .join(silver_links, on="movieId", how="left")
    .join(
        silver_scraped.select("movieId","director","budget_clean","poster_url","scraped_title","status"),
        on="movieId", how="left"
    )
)

silver_movie_master.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}silver_movie_master")
print("silver_movie_master:", silver_movie_master.count())


In [0]:
movie_ids = silver_movie_master.select("movieId").distinct()

orphan_ratings = silver_ratings.join(movie_ids, on="movieId", how="left_anti")
orphan_tags    = silver_tags.join(movie_ids, on="movieId", how="left_anti")

assert orphan_ratings.count() == 0
assert orphan_tags.count() == 0
assert spark.table(f"{PREFIX}silver_movie_master").count() > 0

print("✅ SILVER validated (orphans=0)")
